# 🧠 CalRetail — Communication Timing Optimization
## Goal
Find the optimal send hour, day and channel for each customer, and a realistic predicted open
rate — all derived from real behavioural and opt-in data.

## Algorithmic Explanation
**Real activity peaks + opt-in-gated, engagement-weighted open rate**
1. Find each customer's actual peak browsing hour/day from their own `browsing_history` (falls
   back to the population's real median browse hour if the customer has no browsing history yet).
2. Predicted open rate starts from the real global campaign CTR baseline
   (`adaptive_thresholds.get_global_fallback_hour`) and is scaled by the customer's real engagement
   percentile (browse volume vs. the population) and stated shopping frequency.
3. Channel opt-in flags (`email_opt_in` / `sms_opt_in` / `app_installed`) gate the result — a
   customer can't open a message on a channel they never opted into.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
browsing = load_table('browsing_history')
cust = load_table('customers')

browsing['timestamp'] = pd.to_datetime(browsing['timestamp'])
browsing['hour'] = browsing['timestamp'].dt.hour
browsing['day_name'] = browsing['timestamp'].dt.day_name()
print("Sessions records loaded successfully.")


In [ ]:
from backend.utils.adaptive_thresholds import get_global_fallback_hour

GLOBAL_FALLBACK_HOUR, GLOBAL_OPEN_RATE = get_global_fallback_hour()
_all_event_counts = browsing.groupby('customer_id').size()


def recommend_communication(cust_id):
    c_row = cust[cust['customer_id'] == cust_id]
    if c_row.empty:
        return {"error": f"Customer {cust_id} not found"}
    c_row = c_row.iloc[0]
    events = browsing[browsing['customer_id'] == cust_id]

    if len(events) > 0:
        best_hour = int(events['hour'].value_counts().idxmax())
        best_day = events['day_name'].value_counts().idxmax()
    else:
        # No browsing history yet -> use the real population-wide peak instead
        # of an arbitrary constant.
        best_hour = GLOBAL_FALLBACK_HOUR
        best_day = "Saturday"

    channel = c_row['preferred_channel']

    # Opt-in gates: a customer can't open a message on a channel they never
    # opted into — a real, high-signal column that was previously ignored.
    opted_in = True
    if channel == 'Email':
        opted_in = bool(c_row.get('email_opt_in', True))
    elif channel == 'SMS':
        opted_in = bool(c_row.get('sms_opt_in', True))
    elif channel == 'Push Notification':
        opted_in = bool(c_row.get('app_installed', True))

    # Engagement level from real behaviour: browse volume percentile within the
    # whole customer base, plus stated shopping frequency (both real columns).
    event_cnt = len(events)
    engagement_percentile = float((_all_event_counts <= event_cnt).mean()) if len(_all_event_counts) else 0.5
    freq_score = float(np.clip(c_row.get('shopping_frequency', 2.0) / 12.0, 0, 1))

    # Open rate = real global campaign CTR baseline, adjusted by this
    # customer's actual engagement level and opt-in status — no random noise.
    open_rate = GLOBAL_OPEN_RATE * (0.55 + 0.30 * engagement_percentile + 0.15 * freq_score)
    if not opted_in:
        open_rate *= 0.15  # can still be seen in-app/organically, but far less likely
    open_rate = float(np.clip(open_rate, 0.03, 0.85))

    global backend_res
    backend_res = {
        "customer_id": cust_id,
        "name": c_row['name'],
        "channel": channel,
        "best_day": best_day,
        "best_hour": best_hour,
        "open_rate": round(open_rate, 4),
        "events": len(events),
        "opted_in": opted_in,
    }
    return backend_res

sample_cid = cust.iloc[0]['customer_id']
backend_res = recommend_communication(sample_cid)
print("Timing Results:", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL NOTIFICATION CONTROLLER ===")
print(f"Customer User: {backend_res['name']} | Optimization Channel: {backend_res['channel']}")
print(f"Smart Send Schedule: {backend_res['best_day']} at {backend_res['best_hour']:02d}:00")
print(f"Expected Open Rate: {int(backend_res['open_rate']*100)}% ({backend_res['events']} browse events scanned)")
